# 05 · Dịch đúng ý trước, viết lời để đọc sau
Một dòng phụ đề có hai đáp án: **bản dịch đủ nghĩa** và **lời diễn vừa thời lượng**. Dùng hai adapter độc lập trên cùng NLLB: `meaning_zh` học Trung → Việt không ép ngắn; `spoken_vi` chỉ nhìn bản Việt đã duyệt và ngân sách âm tiết. VieNeu-TTS đọc kết quả; notebook không huấn luyện hay nhân bản chất giọng.

**Run All mặc định chỉ xem** lượt thử đã đo trên máy, không train, không tải model, không ghi dữ liệu. Muốn cải thiện theo phim của mình: khai báo một JSONL đã duyệt ở ô dưới, xem cách chia tập, rồi bật `DO_TRAIN`. Tệp mới luôn nằm ngoài thư mục model đang dùng. Xem [hướng dẫn và kết quả lượt đầu](DIALOGUE.md).

In [ ]:
import json, os, subprocess, sys
from difflib import SequenceMatcher
from pathlib import Path

here = Path(os.environ.get('VSUB_REPO_ROOT', Path.cwd())).resolve()
ROOT = next((d for d in (here, *here.parents) if (d / 'training/train_nllb_lora.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Mở notebook từ repo VSub Studio hoặc đặt VSUB_REPO_ROOT.')
sys.path.insert(0, str(ROOT))
DATA = ROOT / 'data'
MODELS = Path(os.environ.get('VSUB_MODEL_DIR', DATA / 'models')).resolve()
PILOT = MODELS / 'dialogue_lab_v1'
TRAIN_PY = Path(os.environ.get('VSUB_TRAIN_PY') or next((str(DATA / '_train_venv' / p) for p in ('Scripts/python.exe', 'bin/python') if (DATA / '_train_venv' / p).is_file()), sys.executable))

LANG = 'zh'
REVIEWED = None  # Ví dụ: Path('D:/du_lieu_da_duyet/hoi_thoai.jsonl'); không dùng jobs/ hoặc tệp audio
RUN = DATA / 'dialogue_runs' / 'reviewed_01'  # đổi tên cho mỗi thử nghiệm mới
EPOCHS, BATCH, SEED = 3, 8, 42
DO_TRAIN = False
FINAL_TEST = False  # chỉ bật sau khi đã chọn cấu hình trên dev và duyệt tay; chạy đúng một lần
print('Chế độ:', 'train trên GPU' if DO_TRAIN else 'xem', '| Python huấn luyện:', TRAIN_PY)

## 1 · Nhãn phải là lời đã duyệt
Mỗi dòng JSONL là **một câu**. `film` là mã ẩn danh ổn định cho cả bộ phim, không phải số thứ tự câu. `src` là lời gốc; `tgt` là phụ đề Việt đầy đủ; `spoken` là cách diễn viên thực sự muốn đọc (để bằng `tgt` nếu không cần sửa); `cap` là ngân sách 1–60 âm tiết dựa trên khung giờ và tốc độ đọc, **không lấy bằng độ dài lời tham chiếu lúc kiểm tra**. Có thể thêm `checks`, ví dụ `[["đừng", "chớ"], ["vé"]]`, để kiểm tra các từ/cụm quan trọng và từ đồng nghĩa. Lưu cả câu mà máy sửa đúng lẫn câu máy làm sai; đừng coi kết quả máy là nhãn người duyệt.

```json
{"film":"ga-dem","src_lang":"zh","tgt_lang":"vi","src":"别急着换车票。","tgt":"Đừng vội đổi vé xe nhé.","spoken":"Đừng vội đổi vé xe.","cap":5}
```
Cần ít nhất **ba phim** để không dùng câu của cùng phim cho cả train và chấm điểm. Bộ chuẩn nhỏ đi kèm chỉ để thử phần mềm, không được trộn lại vào train. Notebook không đọc thư viện thành viên và không gửi lời thoại ra mạng.

In [ ]:
probes = [json.loads(line) for line in (ROOT / 'training/dialogue_holdout.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
splits = None
if REVIEWED is None:
    print(f'Chưa chọn dữ liệu train. Có {len(probes)} câu thử độc lập trong repo.')
else:
    from training.prepare_dialogue import read_jsonl, split_reviewed, write_reviewed_splits
    rows = read_jsonl(Path(REVIEWED))
    splits = split_reviewed(rows, LANG, seed=SEED, protected=probes)
    for part in ('train', 'dev', 'test'):
        batch = splits[part]
        print(part, ':', len(batch), 'câu /', len({r['film'] for r in batch}), 'phim')
    print('Loại trùng hoặc trùng bộ chuẩn:', len(rows) - sum(map(len, splits.values())))

## 2 · Fine-tune hai tác vụ riêng trên GPU
Ở lượt này chỉ học các nhãn `reviewed`, không lấn át chúng bằng hàng nghìn câu rút gọn nhân tạo. Mỗi epoch đo loss trên **phim dev khác phim train** rồi lấy lại checkpoint có loss dev thấp nhất. Bắt đầu dịch từ adapter hiện hành; bắt đầu lời đọc từ adapter Việt → Việt của lượt thí điểm nếu có. Có thể bỏ warm start để học LoRA mới trên NLLB đã cache. Nếu đổi dữ liệu/LR, đặt `RUN` mới và giữ nguyên phim test để chấm sau cùng.

Cần cache NLLB-600M cùng `torch` CUDA, `transformers`, `peft`, `datasets`, `accelerate`, `sacrebleu` trong `TRAIN_PY` (xem [README](README.md)). Kernel Jupyter không nhất thiết dùng cùng Python. Không tự cài gói hay thay môi trường app.

In [ ]:
env = {**os.environ, 'HF_HUB_OFFLINE':'1', 'TRANSFORMERS_OFFLINE':'1', 'HF_DATASETS_OFFLINE':'1',
       'HF_HUB_DISABLE_TELEMETRY':'1', 'TOKENIZERS_PARALLELISM':'false', 'PYTHONUTF8':'1', 'VSUB_MODEL_DIR':str(MODELS)}
def run(*argv):
    subprocess.run([str(TRAIN_PY), *map(str, argv)], cwd=ROOT, env=env, check=True)

corpus = RUN / 'corpus'
if DO_TRAIN:
    if splits is None or RUN.exists():
        raise ValueError('Chọn dữ liệu đã duyệt và thư mục RUN chưa tồn tại; không ghi đè lượt cũ.')
    run('-c', "import torch, peft, datasets, transformers, accelerate, sacrebleu; assert torch.cuda.is_available(), 'Cần GPU CUDA trong TRAIN_PY'")
    write_reviewed_splits(corpus, splits, LANG)
    legacy = MODELS / ('nllb_lora_dangthoi' + ('' if LANG == 'zh' else '_' + LANG))
    for task, label, initial, lr in [('meaning', 'meaning_' + LANG, legacy, 3e-5), ('spoken', 'spoken_vi', PILOT / 'spoken_vi', 5e-5)]:
        warm = ['--init-adapter', initial] if (initial / 'adapter_model.safetensors').is_file() else []
        run('training/train_nllb_lora.py', '--task', task, '--src', LANG, '--out', RUN / label,
            '--corpus', corpus / f'{task}_train.jsonl', '--dev-corpus', corpus / f'{task}_dev.jsonl',
            '--origins', 'reviewed', '--select-best', '--epochs', EPOCHS, '--batch', BATCH, '--accum', 2,
            '--lr', lr, '--seed', SEED, *warm)
        meta = json.loads((RUN / label / 'train_meta.json').read_text(encoding='utf-8'))
        print(label, 'checkpoint đã chọn:', meta['selected_checkpoint'], '| dev loss theo bước:', meta['eval_loss'])
else:
    print('Không thay đổi model nào.')

## 3 · Đo trên dev, không chỉ đọc loss
So lần lượt adapter dịch mới với NLLB đang dùng, và bản giữ nguyên / luật / model lời đọc thô / luật + model có bộ lọc. Phần lời đọc dùng **cùng bản Việt đã duyệt** ở mọi cột. `chrF` là độ giống tham chiếu theo cụm ký tự; `lexical_matched` là số nhóm thuật ngữ khớp, không phải điểm hiểu nghĩa; `fits` chỉ đo ngân sách âm tiết, không thay cho nghe WAV. Nếu chưa cài adapter hiện hành của `LANG`, mốc so dịch là NLLB gốc đã cache.

In [ ]:
def evaluate(part):
    run('-m', 'training.eval_dialogue', '--src', LANG, '--meaning', RUN / ('meaning_' + LANG),
        '--spoken', RUN / 'spoken_vi', '--holdout', corpus / (part + '.jsonl'), '--out', RUN / (part + '_report.json'))

if DO_TRAIN:
    evaluate('dev')
report_file = RUN / 'dev_report.json'
if not report_file.is_file():
    report_file = PILOT / 'eval.json'
report = json.loads(report_file.read_text(encoding='utf-8')) if report_file.is_file() else None
if report:
    print('Đang xem:', report_file)
    mt = report['mt']
    for ds, old in mt['current'].items():
        new = mt['candidate'][ds]
        terms = f"; nhóm từ {old['lexical_matched']} → {new['lexical_matched']}/{new['lexical_total']}" if new['lexical_total'] else ''
        print(f"{ds} ({old['n']} câu): chrF {old['chrf']:.2f} → {new['chrf']:.2f}{terms}")
    for variant, s in report['spoken'].items():
        print(f"{variant:16} chrF {s['chrf']:.2f} | vừa ngân sách {s['fits']}/{s['n']} | lệch dấu hiệu cần giữ {s['anchor_mismatch']}")
else:
    print('Chưa có báo cáo cục bộ. Xem DIALOGUE.md để tái lập lượt đầu hoặc cung cấp phim đã duyệt.')

### Đọc kết quả qua ba biểu đồ
Biểu đồ 01 so chrF và mức thay đổi trên **từng tập**, không gộp hai tập khác thể loại. Biểu đồ 02 dùng cùng bản Việt đã duyệt; đạt ngân sách âm tiết không đồng nghĩa với đọc vừa số giây, và lệch dấu hiệu chỉ là cảnh báo từ bộ lọc. Biểu đồ 03 lấy loss từ checkpoint của **chính lượt đang xem**. Hai trục loss có thang riêng; điểm hình thoi là lần đo trên dev, không nối thành một đường suy đoán.

In [ ]:
if report:
    try:
        from training import dialogue_charts as charts
    except ModuleNotFoundError as exc:
        if exc.name != 'matplotlib':
            raise
        print('Cần matplotlib trong kernel Jupyter để vẽ. Bảng số liệu phía trên vẫn dùng được.')
    else:
        from IPython.display import display
        from matplotlib import pyplot as plt
        for fig in (charts.translation(report), charts.spoken(report), charts.losses(report_file.parent, report['src'])):
            if fig is None:
                print('Không có lịch sử loss cho lượt này; không vẽ dữ liệu giả.')
            else:
                display(fig)
                plt.close(fig)

In [ ]:
# Xem cả ca lùi và ca tiến; độ giống chữ chỉ để xếp thứ tự đọc, không phải chấm đúng/sai.
def similarity(a, b):
    return SequenceMatcher(None, a.casefold(), b.casefold()).ratio()

if report:
    for before, after, ref in [('current', 'candidate', 'tgt'), ('rules', 'rules_plus_model', 'spoken')]:
        changes = [r for r in report['examples'] if r[before] != r[after]]
        rank = sorted(changes, key=lambda r: similarity(r[after], r[ref]) - similarity(r[before], r[ref]))
        print(f'\n{before} → {after}: {len(rank)} câu đổi')
        for r in (rank if len(rank) <= 4 else rank[:2] + rank[-2:]):
            print('Gốc:', r['src'], '| Cũ:', r[before], '| Mới:', r[after], '| Người duyệt:', r[ref])
    old, new = report['mt']['current'], report['mt']['candidate']
    mt_pass = all(new[k][m] >= s[m] for k, s in old.items() for m in ('chrf','numbers_correct','lexical_matched')) and any(new[k]['chrf'] > s['chrf'] for k, s in old.items())
    rules, voice = (report['spoken'][k] for k in ('rules', 'rules_plus_model'))
    spoken_pass = voice['fits'] > rules['fits'] and voice['chrf'] >= rules['chrf'] and voice['anchor_mismatch'] == 0
    print('Đủ tín hiệu để duyệt thử (không tự triển khai): dịch =', mt_pass, '| lời đọc =', spoken_pass)

## 4 · Đóng băng lựa chọn rồi mới mở phim test
Chỉ bật `FINAL_TEST` sau khi đọc các câu dev sai, đặc biệt tên vai, phủ định và số lượng. So lời ghép với video và nghe cùng một giọng ở cùng tốc độ trước khi quyết định triển khai từng adapter. Nếu nhìn test rồi sửa cấu hình, cần **phim test mới** cho kết luận tiếp theo. Hãy đưa các lỗi dev đã được biên tập viên sửa vào lượt huấn luyện sau, không đưa phim test vào. Không có cell nào tự bật model cho studio.

In [ ]:
if FINAL_TEST:
    if not (RUN / 'dev_report.json').is_file() or (RUN / 'test_report.json').exists():
        raise ValueError('Cần báo cáo dev của lượt này; không ghi đè báo cáo test đã có.')
    evaluate('test')
    blind = json.loads((RUN / 'test_report.json').read_text(encoding='utf-8'))
    print(json.dumps({'mt': blind['mt'], 'spoken': blind['spoken']}, indent=2, ensure_ascii=False))
else:
    print('Chưa mở phim test của lượt train mới.')